# Task 6 - OpenCV       
Goal: Use opencv to help Tim consitently track the ball.         
Deadline: 15th March    

In [448]:
import numpy as np 
import cv2 as cv
import math
from collections import Counter

In [ ]:
cap = cv.VideoCapture('data/Volleyball.mp4')

# Kernels
kernel_open = np.ones((3,3), np.uint8)
kernel_close = np.ones((7,7), np.uint8)
kernel_small = np.ones((5,5), np.uint8)

total_yellow = 0
total_blue = 0

yellow_counts = []
blue_counts = []

frame_count_yellow = 0
frame_count_blue = 0

prev_center = None
prev_radius = None
path = []

lock_counter = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    height = frame.shape[0]
    blue_total_frame = 0

    # Convert to HSV
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    # MASKS 
    # Ball (tight yellow)
    mask_ball = cv.inRange(hsv, (15,120,120), (40,255,255))

    # Yellow team (wider)
    mask_yellow = cv.inRange(hsv, (10,100,100), (40,255,255))

    # Blue team
    mask_blue = cv.inRange(hsv, (110,50,60), (145,255,255))

    # Blue team, white libero
    mask_white = cv.inRange(hsv, (0, 0, 200), (180, 50, 255))

    # MORPHOLOGY
    def clean(mask):
        opened = cv.morphologyEx(mask, cv.MORPH_OPEN, kernel_open)
        closed = cv.morphologyEx(opened, cv.MORPH_CLOSE, kernel_close)
        return closed

    clean_yellow = clean(mask_yellow)
    clean_blue = clean(mask_blue)
    clean_white = clean(mask_white)
    ball_dilate = cv.dilate(mask_ball, kernel_small)
    clean_ball = cv.morphologyEx(ball_dilate, cv.MORPH_CLOSE, kernel_close)

    # Counting yellow team players
    contours_yellow, _ = cv.findContours(clean_yellow, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    yellow_count = 0

    for cnt in contours_yellow:
        area = cv.contourArea(cnt)

        if 500 < area < 5000: 
            yellow_count += 1
            cv.drawContours(frame, [cnt], -1, (0, 255, 255), 2)

    # Counting blue team players
    contours_blue, _ = cv.findContours(clean_blue, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    blue_count = 0

    for cnt in contours_blue:
        area = cv.contourArea(cnt)
        
        if 500 < area < 5000: 
            x, y, w, h = cv.boundingRect(cnt)

            if y > height * 0.4:
                blue_count += 1
                cv.drawContours(frame, [cnt], -1, (255, 0, 0), 2)

    # Counting liberos for each team (white uniform)
    contours_white, _ = cv.findContours(clean_white, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    blue_total_frame = blue_count
    for cnt in contours_white:
        area = cv.contourArea(cnt)

        if 800 < area < 4000:
            x, y, w, h = cv.boundingRect(cnt)
            aspect_ratio = h / float(w)

            if 1.2 < aspect_ratio < 4 and w < frame.shape[1] * 0.2:
                if y > height * 0.4:
                    blue_total_frame += 1
                    cv.drawContours(frame, [cnt], -1, (255, 255, 255), 2)
                else:
                    yellow_count += 1
                    cv.drawContours(frame, [cnt], -1, (255, 255, 255), 2)

    # Counting total number of players per team in video + storing per frame count
    if blue_total_frame > 0:
        total_blue += blue_total_frame
        blue_counts.append(blue_total_frame)
        frame_count_blue += 1

    if yellow_count > 0:
        total_yellow += yellow_count
        yellow_counts.append(yellow_count)
        frame_count_yellow += 1

    # TRACKING BALL
    # Find contours
    contours_ball, _ = cv.findContours(clean_ball, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    best_center = None
    best_radius = 0

    # tracking metric
    best_metric = -1

    for cnt in contours_ball:
        area = cv.contourArea(cnt)
        perimeter = cv.arcLength(cnt, True)

        x, y, w, h = cv.boundingRect(cnt)
        aspect_ratio = w / float(h)

        if aspect_ratio < 0.6 or aspect_ratio > 1.4:
            continue
        
        if perimeter == 0:
            continue

        circularity = 4 * math.pi * area / (perimeter ** 2)

        # BALL FILTER
        if 25 < area < 400 and circularity > 0.35:

            (x, y), radius = cv.minEnclosingCircle(cnt)
            if radius < 4 or radius > 13:
                continue

            center = (int(x), int(y))

            if prev_center is None:
                # Only initialize with strong candidate near center-ish region
                if area > 80 and circularity > 0.5:
                    score = circularity * area
                else:
                    continue
                if score > best_metric:
                    best_metric = score
                    best_center = center
                    best_radius = int(radius)
            else:
                dx = center[0] - prev_center[0]
                dy = center[1] - prev_center[1]
                dist_sq = dx*dx + dy*dy
                if dist_sq < 10 or dist_sq > 10000:   # too little movement → likely stuck on noise
                    continue

                # ignore too far 
                if lock_counter > 5 and dist_sq > 10000:
                    continue

                # COMBINED SCORE
                score = (1 / (dist_sq + 1)) * circularity * area * radius

                if score > best_metric and score > 0.002:
                    best_metric = score
                    best_center = center
                    best_radius = int(radius)

    # Draw result
    if best_center is not None:
        cv.circle(frame, best_center, best_radius, (0, 255, 0), 3)  # green circle
        cv.circle(frame, best_center, 3, (0, 0, 255), -1)           # red center dot
        print(f"Ball at: {best_center}, radius: {best_radius}")

        # only trust if movement is reasonable
        if prev_center is not None and best_radius > 0:
            dx = best_center[0] - prev_center[0]
            dy = best_center[1] - prev_center[1]
            dist = (dx*dx + dy*dy)**0.5

            if prev_radius is not None:
                if abs(best_radius - prev_radius) > 5:
                    continue

            if 10 < dist < 85:   
                path.append(best_center)
        else:
            path.append(best_center)

        prev_center = best_center
        prev_radius = best_radius
        lock_counter +=1
    else:
        lock_counter -= 1

    if lock_counter < -3:
        prev_center = None
        prev_radius = None
        lock_counter = 0

    # Limit path size
    if len(path) > 20:
        path.pop(0)

    # Draw trajectory
    #for i in range(1, len(path)):
    #    if path[i-1] is None or path[i] is None:
    #        continue
    #    cv.line(frame, path[i-1], path[i], (0, 255, 0), 2)

    # Convert to BGR for display
    ball_bgr = cv.cvtColor(clean_ball, cv.COLOR_GRAY2BGR)
    yellow_bgr = cv.cvtColor(clean_yellow, cv.COLOR_GRAY2BGR)
    blue_bgr = cv.cvtColor(clean_blue, cv.COLOR_GRAY2BGR)

    # Masks labels
    cv.putText(ball_bgr, "Ball Mask", (10, 25),
           cv.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv.putText(yellow_bgr, "Yellow Mask", (10, 25),
            cv.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

    cv.putText(blue_bgr, "Blue Mask", (10, 25),
            cv.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    
    # DISPLAY 
    top = cv.hconcat([frame, ball_bgr])
    bottom = cv.hconcat([yellow_bgr, blue_bgr])
    combined = cv.vconcat([top, bottom])

    # Yellow text + background
    overlay = combined.copy()
    cv.rectangle(overlay, (10, 10), (300, 120), (255, 255, 255), -1)
    alpha = 0.4
    combined = cv.addWeighted(overlay, alpha, combined, 1 - alpha, 0)
    cv.putText(combined, f"Yellow Players: {yellow_count}", (20, 45),
                cv.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    # Blue text 
    cv.putText(combined, f"Blue Players: {blue_total_frame}", (20, 105),
            cv.FONT_HERSHEY_SIMPLEX, 1, (255,0,0), 2)
    
    cv.imshow("Pipeline View", combined)

    if cv.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv.destroyAllWindows()

# Calculating and displaying average and most common no of players detected
if frame_count_yellow > 0:
    avg_yellow = total_yellow / frame_count_yellow
    print("Average Yellow Players:", avg_yellow)

if frame_count_blue > 0:
    avg_blue = total_blue/ frame_count_blue
    print("Average Blue Players:", avg_blue)

yellow_mode = Counter(yellow_counts).most_common(1)[0][0]
blue_mode = Counter(blue_counts).most_common(1)[0][0]

print("Yellow Mode:", yellow_mode)
print("Blue Mode:", blue_mode)

Ball at: (625, 325), radius: 10
Ball at: (625, 324), radius: 9
Ball at: (535, 343), radius: 6
Ball at: (625, 324), radius: 9
Ball at: (564, 345), radius: 5
Ball at: (625, 324), radius: 9
Ball at: (564, 345), radius: 5
Ball at: (625, 324), radius: 9
Ball at: (564, 345), radius: 5
Ball at: (562, 342), radius: 7
Ball at: (625, 323), radius: 9
Ball at: (562, 342), radius: 8
Ball at: (624, 323), radius: 8
Ball at: (562, 342), radius: 8
Ball at: (537, 342), radius: 7
Ball at: (624, 324), radius: 9
Ball at: (627, 323), radius: 7
Ball at: (626, 322), radius: 8
Ball at: (621, 325), radius: 7
Ball at: (624, 322), radius: 7
Ball at: (627, 323), radius: 7
Ball at: (563, 340), radius: 9
Ball at: (539, 341), radius: 9
Ball at: (563, 340), radius: 9
Ball at: (539, 341), radius: 9
Ball at: (563, 340), radius: 9
Ball at: (541, 341), radius: 7
Ball at: (563, 341), radius: 9
Ball at: (538, 341), radius: 8
Ball at: (562, 340), radius: 9
Ball at: (538, 340), radius: 6
Ball at: (561, 340), radius: 9
Ball at

## Result: 
Average Yellow Players: 5.47    
Average Blue Players: 5.32   
Yellow Mode: 6  
Blue Mode: 6    